# Sesión 3A · Leer y limpiar datos con pandas

**La pregunta de hoy:** ¿en qué distritos del Perú golpea más el dengue?

Vamos a responderla con datos reales del MINSA. No hay teoría: cada celda
resuelve un pedazo de la pregunta.

In [2]:
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

## 1. Abrir el archivo

`read_csv` es la puerta de entrada. El 90% del trabajo empieza aquí.

Ojo al warning que aparece: pandas avisa que una columna tiene "tipos mixtos".
Es la primera pista de que estos datos vienen sucios.

In [10]:
dengue = pd.read_csv("../../data/03-pandas/dengue_distritos.csv", encoding="utf-8-sig")
dengue.head(5)

/var/folders/9r/dv5kdqsn3jj_s4vlrjy493p00000gn/T/ipykernel_86720/3000841712.py:1: DtypeWarning: Columns (0: Casos) have mixed types. Specify dtype option on import or set low_memory=False.
  dengue = pd.read_csv("../../data/03-pandas/dengue_distritos.csv", encoding="utf-8-sig")


,Año,Semana,Eventos o daños,Departamento,Distrito,Provincia,Ubigeo,Casos
0,2020,43,Dengue,CUSCO,MEGANTONI,LA CONVENCION,80914,2.0
1,2021,41,Dengue,MOQUEGUA,MOQUEGUA,MARISCAL NIETO,180101,NaN
2,2021,42,Dengue,MOQUEGUA,MOQUEGUA,MARISCAL NIETO,180101,NaN
3,2020,1,Dengue,CUSCO,MEGANTONI,LA CONVENCION,80914,0.0
4,2020,53,Dengue,CUSCO,MEGANTONI,LA CONVENCION,80914,2.0


### ¿Qué tan grande es? `shape` devuelve (filas, columnas)

In [11]:
dengue.shape

(172144, 8)

### `info()` es lo primero que uno mira siempre

Dice cuántos valores no nulos tiene cada columna y de qué tipo es.
Fíjate en `Casos`: dice `object`, o sea **texto**. Eso es un problema.

In [15]:
dengue

,Año,Semana,Eventos o daños,Departamento,Distrito,Provincia,Ubigeo,Casos
0,2020,43,Dengue,CUSCO,MEGANTONI,LA CONVENCION,80914,2.0
1,2021,41,Dengue,MOQUEGUA,MOQUEGUA,MARISCAL NIETO,180101,NaN
2,2021,42,Dengue,MOQUEGUA,MOQUEGUA,MARISCAL NIETO,180101,NaN
3,2020,1,Dengue,CUSCO,MEGANTONI,LA CONVENCION,80914,0.0
4,2020,53,Dengue,CUSCO,MEGANTONI,LA CONVENCION,80914,2.0
...,...,...,...,...,...,...,...,...
172139,2021,49,Dengue,MADRE DE DIOS,MADRE DE DIOS,MANU,170203,NaN
172140,2021,49,Dengue,MADRE DE DIOS,TAHUAMANU,TAHUAMANU,170303,NaN
172141,2021,50,Dengue,MADRE DE DIOS,MADRE DE DIOS,MANU,170203,NaN
172142,2021,50,Dengue,MADRE DE DIOS,TAHUAMANU,TAHUAMANU,170303,NaN


In [14]:
dengue.info()

<class 'pandas.DataFrame'>
RangeIndex: 172144 entries, 0 to 172143
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   Año              172144 non-null  int64 
 1   Semana           172144 non-null  int64 
 2   Eventos o daños  172144 non-null  str   
 3   Departamento     172144 non-null  str   
 4   Distrito         172144 non-null  str   
 5   Provincia        172144 non-null  str   
 6   Ubigeo           172144 non-null  int64 
 7   Casos            166047 non-null  object
dtypes: int64(3), object(1), str(4)
memory usage: 15.5+ MB


In [12]:
dengue.info()

<class 'pandas.DataFrame'>
RangeIndex: 172144 entries, 0 to 172143
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   Año              172144 non-null  int64 
 1   Semana           172144 non-null  int64 
 2   Eventos o daños  172144 non-null  str   
 3   Departamento     172144 non-null  str   
 4   Distrito         172144 non-null  str   
 5   Provincia        172144 non-null  str   
 6   Ubigeo           172144 non-null  int64 
 7   Casos            166047 non-null  object
dtypes: int64(3), object(1), str(4)
memory usage: 15.5+ MB


## 2. Ordenar los nombres de las columnas

Columnas con tildes, espacios y mayúsculas hacen la vida imposible.
Se renombran una sola vez, al inicio.

In [23]:
dengue.info()

<class 'pandas.DataFrame'>
RangeIndex: 172144 entries, 0 to 172143
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   anio          172144 non-null  int64 
 1   semana        172144 non-null  int64 
 2   evento        172144 non-null  str   
 3   departamento  172144 non-null  str   
 4   distrito      172144 non-null  str   
 5   provincia     172144 non-null  str   
 6   ubigeo        172144 non-null  int64 
 7   casos         166047 non-null  object
dtypes: int64(3), object(1), str(4)
memory usage: 15.5+ MB


In [17]:
dengue = dengue.rename(
    columns={
        "Año": "anio",
        "Semana": "semana",
        "Eventos o daños": "evento",
        "Departamento": "departamento",
        "Provincia": "provincia",
        "Distrito": "distrito",
        "Ubigeo": "ubigeo",
        "Casos": "casos",
    }
)
dengue.columns

Index(['anio', 'semana', 'evento', 'departamento', 'distrito', 'provincia', 'ubigeo', 'casos'], dtype='str')

## 3. Los tres problemas de estos datos

### Problema 1: `casos` es texto, no número

No se puede sumar texto. `to_numeric` lo convierte; `errors="coerce"`
transforma en nulo lo que no se pueda convertir en vez de reventar.

In [25]:
dengue["casos"].head()

0    2.0
1    NaN
2    NaN
3    0.0
4    2.0
Name: casos, dtype: object

In [ ]:
dengue["casos"] = pd.to_numeric(dengue["casos"], errors="coerce")
dengue["casos"].describe()

0         2.0
1         NaN
2         NaN
3         0.0
4         2.0
         ... 
172139    NaN
172140    NaN
172141    NaN
172142    NaN
172143    NaN
Name: casos, Length: 172144, dtype: float64

In [29]:
dengue["casos"].describe()

count    166040.000000
mean          1.391966
std          11.290563
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         912.000000
Name: casos, dtype: float64

### Problema 2: hay nulos

¿Cuántos? `isna().sum()` cuenta nulos por columna.

In [31]:
dengue.isna().sum()

anio               0
semana             0
evento             0
departamento       0
distrito           0
provincia          0
ubigeo             0
casos           6104
dtype: int64

En este caso un nulo significa "esa semana no se reportaron casos",
así que vale cero.

In [32]:
dengue["casos"] = dengue["casos"].fillna(0)
dengue.isna().sum()

anio            0
semana          0
evento          0
departamento    0
distrito        0
provincia       0
ubigeo          0
casos           0
dtype: int64

### Problema 3: el ubigeo perdió el cero de la izquierda

El ubigeo de Megantoni es `080914`, pero al leerse como número quedó `80914`.
Este error rompe cualquier cruce con otra base. Se arregla convirtiendo a
texto y rellenando con ceros hasta 6 dígitos.

In [33]:
dengue["ubigeo"].head()

0     80914
1    180101
2    180101
3     80914
4     80914
Name: ubigeo, dtype: int64

In [34]:
dengue["ubigeo"] = dengue["ubigeo"].astype(str).str.zfill(6)
dengue["ubigeo"].head()

0    080914
1    180101
2    180101
3    080914
4    080914
Name: ubigeo, dtype: str

## 4. Duplicados

`duplicated()` marca las filas repetidas. Aquí no hay ninguna, pero esto se
revisa **siempre**: una fila duplicada infla los totales sin que uno lo note.

In [35]:
dengue.duplicated().sum()

np.int64(0)

In [36]:
dengue = dengue.drop_duplicates()
dengue.shape

(172144, 8)

## 5. Mirar los datos por dentro

`value_counts()` es la herramienta más útil de pandas para datos categóricos.

In [15]:
dengue["departamento"].value_counts().head(10)

departamento
SAN MARTIN     27825
PIURA          22949
LORETO         17808
ICA            13727
LAMBAYEQUE     11872
LA LIBERTAD    10388
CAJAMARCA       9540
HUANUCO         7632
LIMA            7420
AMAZONAS        6572
Name: count, dtype: int64

In [38]:
dengue["evento"].value_counts()

evento
Dengue    172144
Name: count, dtype: int64

In [39]:
dengue["anio"].value_counts().sort_index()

anio
2015    24168
2016    24433
2017    24645
2018    24645
2019    24645
2020    24751
2021    24857
Name: count, dtype: int64

## 6. Filtrar filas

Dentro de los corchetes va una condición. El resultado es una tabla nueva.

In [42]:
loreto = dengue[dengue["departamento"] == "LORETO"] 
loreto

,anio,semana,evento,departamento,distrito,provincia,ubigeo,casos
826,2021,41,Dengue,LORETO,PADRE MARQUEZ,UCAYALI,160603,0.0
827,2021,42,Dengue,LORETO,PADRE MARQUEZ,UCAYALI,160603,0.0
828,2021,43,Dengue,LORETO,PADRE MARQUEZ,UCAYALI,160603,0.0
829,2021,44,Dengue,LORETO,PADRE MARQUEZ,UCAYALI,160603,0.0
830,2021,45,Dengue,LORETO,PADRE MARQUEZ,UCAYALI,160603,0.0
...,...,...,...,...,...,...,...,...
171579,2018,17,Dengue,LORETO,NAUTA,LORETO,160301,4.0
171580,2018,34,Dengue,LORETO,ANDOAS,DATEM DEL MARAÑON,160706,3.0
171588,2018,51,Dengue,LORETO,YURIMAGUAS,ALTO AMAZONAS,160201,41.0
171946,2020,38,Dengue,LORETO,CONTAMANA,UCAYALI,160601,10.0


In [ ]:
loreto = dengue[dengue["departamento"] == "LORETO"] 
loreto.shape

(17808, 8)

### Dos condiciones a la vez

Cada condición va entre paréntesis. `&` es "y", `|` es "o".

In [46]:
loreto_2021 = dengue[(dengue["departamento"] == "LORETO") & (dengue["anio"] == 2021)]
loreto_2021.reset_index(drop=True, inplace=True)
loreto_2021

,anio,semana,evento,departamento,distrito,provincia,ubigeo,casos
0,2021,41,Dengue,LORETO,PADRE MARQUEZ,UCAYALI,160603,0.0
1,2021,42,Dengue,LORETO,PADRE MARQUEZ,UCAYALI,160603,0.0
2,2021,43,Dengue,LORETO,PADRE MARQUEZ,UCAYALI,160603,0.0
3,2021,44,Dengue,LORETO,PADRE MARQUEZ,UCAYALI,160603,0.0
4,2021,45,Dengue,LORETO,PADRE MARQUEZ,UCAYALI,160603,0.0
...,...,...,...,...,...,...,...,...
2539,2021,52,Dengue,LORETO,ANDOAS,DATEM DEL MARAÑON,160706,0.0
2540,2021,53,Dengue,LORETO,SANTA CRUZ,ALTO AMAZONAS,160210,0.0
2541,2021,53,Dengue,LORETO,TROMPETEROS,LORETO,160304,0.0
2542,2021,53,Dengue,LORETO,PAMPA HERMOSA,UCAYALI,160604,0.0


### `isin` cuando son varias opciones

Evita escribir cinco condiciones con `|`.

In [52]:
selva = dengue[(dengue["departamento"] == "LORETO") & (dengue["departamento"] == "UCAYALI") & (dengue["departamento"] == "MADRE DE DIOS")]
selva.shape

(0, 8)

In [48]:
selva = dengue[dengue["departamento"].isin(["LORETO", "UCAYALI", "MADRE DE DIOS"])]
selva

,anio,semana,evento,departamento,distrito,provincia,ubigeo,casos
826,2021,41,Dengue,LORETO,PADRE MARQUEZ,UCAYALI,160603,0.0
827,2021,42,Dengue,LORETO,PADRE MARQUEZ,UCAYALI,160603,0.0
828,2021,43,Dengue,LORETO,PADRE MARQUEZ,UCAYALI,160603,0.0
829,2021,44,Dengue,LORETO,PADRE MARQUEZ,UCAYALI,160603,0.0
830,2021,45,Dengue,LORETO,PADRE MARQUEZ,UCAYALI,160603,0.0
...,...,...,...,...,...,...,...,...
172139,2021,49,Dengue,MADRE DE DIOS,MADRE DE DIOS,MANU,170203,0.0
172140,2021,49,Dengue,MADRE DE DIOS,TAHUAMANU,TAHUAMANU,170303,0.0
172141,2021,50,Dengue,MADRE DE DIOS,MADRE DE DIOS,MANU,170203,0.0
172142,2021,50,Dengue,MADRE DE DIOS,TAHUAMANU,TAHUAMANU,170303,0.0


### `query` hace lo mismo pero se lee mejor

In [54]:
dengue.query("anio == 2021 and casos > 50")

,anio,semana,evento,departamento,distrito,provincia,ubigeo,casos
47308,2021,23,Dengue,PIURA,CHULUCANAS,MORROPON,200401,143.0
47310,2021,24,Dengue,PIURA,CHULUCANAS,MORROPON,200401,128.0
47311,2021,25,Dengue,PIURA,CHULUCANAS,MORROPON,200401,103.0
47312,2021,26,Dengue,PIURA,CHULUCANAS,MORROPON,200401,111.0
47313,2021,27,Dengue,PIURA,CHULUCANAS,MORROPON,200401,57.0
...,...,...,...,...,...,...,...,...
136952,2021,1,Dengue,LORETO,YURIMAGUAS,ALTO AMAZONAS,160201,221.0
136955,2021,2,Dengue,LORETO,YURIMAGUAS,ALTO AMAZONAS,160201,195.0
136959,2021,3,Dengue,LORETO,YURIMAGUAS,ALTO AMAZONAS,160201,124.0
136961,2021,4,Dengue,LORETO,YURIMAGUAS,ALTO AMAZONAS,160201,119.0


## 7. Seleccionar columnas y filas: `loc` e `iloc`

- `loc` usa **nombres**
- `iloc` usa **posiciones**

In [22]:
dengue.loc[:, ["departamento", "distrito", "anio", "casos"]].head()

,departamento,distrito,anio,casos
0,CUSCO,MEGANTONI,2020,2.0
1,MOQUEGUA,MOQUEGUA,2021,0.0
2,MOQUEGUA,MOQUEGUA,2021,0.0
3,CUSCO,MEGANTONI,2020,0.0
4,CUSCO,MEGANTONI,2020,2.0


In [23]:
dengue.iloc[0:5, 0:4]

,anio,semana,evento,departamento
0,2020,43,Dengue,CUSCO
1,2021,41,Dengue,MOQUEGUA
2,2021,42,Dengue,MOQUEGUA
3,2020,1,Dengue,CUSCO
4,2020,53,Dengue,CUSCO


### `loc` con condición y columnas al mismo tiempo

In [24]:
dengue.loc[dengue["casos"] > 200, ["departamento", "distrito", "anio", "semana", "casos"]]

,departamento,distrito,anio,semana,casos
736,ICA,ICA,2020,15,300.0
739,ICA,ICA,2020,16,444.0
1370,ICA,ICA,2020,17,418.0
1373,ICA,ICA,2020,18,320.0
9071,LORETO,IQUITOS,2020,10,250.0
...,...,...,...,...,...
158137,UCAYALI,CALLERIA,2020,44,256.0
158138,UCAYALI,CALLERIA,2020,45,266.0
158141,UCAYALI,CALLERIA,2020,46,239.0
158143,UCAYALI,CALLERIA,2020,47,211.0


## 8. Crear columnas nuevas

Se escribe una columna que no existe y pandas la crea.

In [55]:
dengue["casos"] > 100

0         False
1         False
2         False
3         False
4         False
          ...  
172139    False
172140    False
172141    False
172142    False
172143    False
Name: casos, Length: 172144, dtype: bool

In [60]:
dengue["brote"] = dengue["casos"] > 10
dengue["brote"].value_counts()

brote
False    167641
True       4503
Name: count, dtype: int64

In [64]:
dengue["sur_peru"] = dengue["departamento"].isin(["ICA", "AREQUIPA", "MOQUEGUA", "TACNA", "PUNO"])
dengue

,anio,semana,evento,departamento,distrito,provincia,ubigeo,casos,brote,sur_peru
0,2020,43,Dengue,CUSCO,MEGANTONI,LA CONVENCION,080914,2.0,False,False
1,2021,41,Dengue,MOQUEGUA,MOQUEGUA,MARISCAL NIETO,180101,0.0,False,True
2,2021,42,Dengue,MOQUEGUA,MOQUEGUA,MARISCAL NIETO,180101,0.0,False,True
3,2020,1,Dengue,CUSCO,MEGANTONI,LA CONVENCION,080914,0.0,False,False
4,2020,53,Dengue,CUSCO,MEGANTONI,LA CONVENCION,080914,2.0,False,False
...,...,...,...,...,...,...,...,...,...,...
172139,2021,49,Dengue,MADRE DE DIOS,MADRE DE DIOS,MANU,170203,0.0,False,False
172140,2021,49,Dengue,MADRE DE DIOS,TAHUAMANU,TAHUAMANU,170303,0.0,False,False
172141,2021,50,Dengue,MADRE DE DIOS,MADRE DE DIOS,MANU,170203,0.0,False,False
172142,2021,50,Dengue,MADRE DE DIOS,TAHUAMANU,TAHUAMANU,170303,0.0,False,False


### Una columna a partir de texto

`.str` da acceso a las operaciones de texto sobre toda la columna.

In [67]:
dengue["departamento"] = (
    dengue["departamento"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)
dengue["departamento"]

0                 Cusco
1              Moquegua
2              Moquegua
3                 Cusco
4                 Cusco
              ...      
172139    Madre De Dios
172140    Madre De Dios
172141    Madre De Dios
172142    Madre De Dios
172143    Madre De Dios
Name: departamento, Length: 172144, dtype: str

## 9. Ordenar

In [71]:
dengue.sort_values("casos", ascending=False).head(10)

,anio,semana,evento,departamento,distrito,provincia,ubigeo,casos,brote,sur_peru
76346,2017,20,Dengue,Piura,PIURA,PIURA,200101,912.0,True,False
76227,2017,16,Dengue,Piura,CASTILLA,PIURA,200104,804.0,True,False
76221,2017,14,Dengue,Piura,CASTILLA,PIURA,200104,742.0,True,False
76335,2017,18,Dengue,Piura,CASTILLA,PIURA,200104,690.0,True,False
76342,2017,14,Dengue,Piura,PIURA,PIURA,200101,688.0,True,False
76343,2017,15,Dengue,Piura,CASTILLA,PIURA,200104,684.0,True,False
76239,2017,20,Dengue,Piura,VEINTISEIS DE OCTUBRE,PIURA,200115,665.0,True,False
76234,2017,18,Dengue,Piura,SULLANA,SULLANA,200601,660.0,True,False
76345,2017,17,Dengue,Piura,CASTILLA,PIURA,200104,654.0,True,False
76229,2017,17,Dengue,Piura,VEINTISEIS DE OCTUBRE,PIURA,200115,611.0,True,False


## 10. Guardar el resultado

Ya tenemos una base limpia. Se guarda para no repetir todo esto.

In [73]:
dengue.to_csv("dengue_limpio.csv", index=False)
dengue.shape

(172144, 10)

---
## Lo que hicimos

| Paso | Código |
|---|---|
| Abrir | `pd.read_csv(ruta)` |
| Inspeccionar | `.shape`, `.head()`, `.info()` |
| Renombrar | `.rename(columns={...})` |
| Texto a número | `pd.to_numeric(col, errors="coerce")` |
| Rellenar nulos | `.fillna(0)` |
| Arreglar ubigeo | `.astype(str).str.zfill(6)` |
| Duplicados | `.duplicated()`, `.drop_duplicates()` |
| Contar categorías | `.value_counts()` |
| Filtrar | `df[cond]`, `.isin()`, `.query()` |
| Seleccionar | `.loc[]`, `.iloc[]` |
| Ordenar | `.sort_values()` |
| Guardar | `.to_csv()` |

**Sigue en el notebook 3B:** agrupar, cruzar con otra base y armar el cuadro final.